# ResNet-50 | Plain BCE | Balanced | Non-Ben Graham | Validation + Test

Train a **new** ImageNet-pretrained ResNet-50 using two-stage fine-tuning and evaluate it on the fixed patient-separated validation and test splits.

- **Epoch 1:** train the new binary classification head only; keep the pretrained backbone and batch-normalization statistics frozen.
- **Epochs 2–12:** unfreeze the whole network; apply smaller learning rates to earlier layers.
- Save the checkpoint with the highest **validation AUROC**. Early stopping (patience 3) applies to the full fine-tuning stage.
- Select a threshold **using validation only** to target at least 90% sensitivity; evaluate the test set once using the fixed saved threshold.

**Before running:** Attach the output of `lohitashwamadhan/02-preprocessing` under Kaggle **Add Input**, enable a GPU and Internet for ImageNet weights (or provide a local compatible weight file). Run all cells in order. No old checkpoint is used.

The BG and non-BG notebooks intentionally share the same settings. Only the image folder differs.


In [1]:
import gc
import json
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, roc_auc_score, roc_curve
)

SEED = 42
VARIANT = 'nonbg'
INPUT_ZIP = Path('/kaggle/input/notebooks/lohitashwamadhan/02-preprocessing/_output_.zip')
DATA_ROOT = Path('/kaggle/working/extracted_data')
OUTPUT_ROOT = Path('/kaggle/working/resnet50_plainbce_balanced_nonbg')
IMAGE_SIZE = 512
BATCH_SIZE = 32
MAX_EPOCHS = 12
PATIENCE = 3
HEAD_LR = 1e-4
EARLY_LR = 1e-5
MIDDLE_LR = 3e-5
LATE_LR = 1e-4
WEIGHT_DECAY = 1e-4
TARGET_SENSITIVITY = 0.90
NUM_WORKERS = 2
LOCAL_IMAGENET_WEIGHTS = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Enable a GPU accelerator in Kaggle settings.'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Device:', device, '| Variant:', VARIANT)


Device: cuda | Variant: nonbg


## Load the preprocessing output

In [2]:
assert INPUT_ZIP.is_file(), f'Missing input: {INPUT_ZIP}. Attach the 02-preprocessing notebook output.'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
image_folder = 'processed_bg' if VARIANT == 'bg' else 'processed'
with zipfile.ZipFile(INPUT_ZIP) as z:
    members = []
    for info in z.infolist():
        member_name = info.filename
        if member_name != 'manifest_v2.csv' and not member_name.startswith(image_folder + '/'):
            continue
        destination = (DATA_ROOT / member_name).resolve()
        assert destination.is_relative_to(DATA_ROOT.resolve()), f'Unsafe path: {member_name}'
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000, f'Symlink in ZIP: {member_name}'
        if not info.is_dir() and not destination.exists():
            members.append(info)
    print('Extracting files:', len(members), flush=True)
    z.extractall(DATA_ROOT, members)
assert (DATA_ROOT / 'manifest_v2.csv').is_file()
print('Data ready:', DATA_ROOT)


Extracting files: 88840
Data ready: /kaggle/working/extracted_data


In [3]:
df = pd.read_csv(DATA_ROOT / 'manifest_v2.csv')
required = {'filename', 'split', 'label', 'group_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
df = df[df['split'].isin(['train', 'val', 'test'])].copy().reset_index(drop=True)
assert df['label'].notna().all() and set(df['label'].astype(int)) == {0, 1}
assert df['group_id'].notna().all(), 'Missing patient IDs'
assert (df.groupby('group_id')['split'].nunique() == 1).all(), 'Patient leakage across data splits'
expected_counts = {'train': {0: 15126, 1: 15161},
                   'val': {0: 4892, 1: 1591},
                   'test': {0: 4907, 1: 1584}}
print(pd.crosstab(df['split'], df['label']))
for split, counts in expected_counts.items():
    for label, expected in counts.items():
        actual = int(((df['split'] == split) & (df['label'] == label)).sum())
        assert actual == expected, f'{split} label {label}: expected {expected}, got {actual}'

paths = [DATA_ROOT / image_folder / split / filename
         for split, filename in zip(df['split'], df['filename'])]
missing = [str(path) for path in paths if not path.is_file()]
assert not missing, f'Missing {len(missing)} images; examples: {missing[:3]}'
print('Images verified:', len(paths))


label      0      1
split              
test    4907   1584
train  15126  15161
val     4892   1591
Images verified: 43261


## Dataset and model

In [4]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class RetinalDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = DATA_ROOT / image_folder / row['split'] / row['filename']
        with Image.open(path) as image:
            x = transform(image.convert('RGB'))
        y = torch.tensor(float(row['label']), dtype=torch.float32)
        return x, y

loaders = {}
for split in ['train', 'val', 'test']:
    subset = df[df['split'] == split].copy()
    loaders[split] = DataLoader(
        RetinalDataset(subset),
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

def initialize_model():
    if LOCAL_IMAGENET_WEIGHTS:
        model = models.resnet50(weights=None)
        state = torch.load(LOCAL_IMAGENET_WEIGHTS, map_location='cpu', weights_only=True)
        if 'state_dict' in state:
            state = state['state_dict']
        model.load_state_dict(state, strict=True)
    else:
        try:
            model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        except Exception as exc:
            raise RuntimeError(
                'ImageNet weights unavailable. Enable Kaggle Internet or supply LOCAL_IMAGENET_WEIGHTS.'
            ) from exc
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model.to(device)

model = initialize_model()
criterion = nn.BCEWithLogitsLoss()
print('Model initialized with a new binary classification head.')


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 175MB/s]


Model initialized with a new binary classification head.


## Fine-tuning protocol

In [5]:
def set_training_stage(model, stage):
    if stage == 'head':
        for parameter in model.parameters():
            parameter.requires_grad = False
        for parameter in model.fc.parameters():
            parameter.requires_grad = True
        return torch.optim.AdamW(model.fc.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)

    for parameter in model.parameters():
        parameter.requires_grad = True
    groups = [
        {'params': list(model.conv1.parameters()) + list(model.bn1.parameters()) +
                   list(model.layer1.parameters()), 'lr': EARLY_LR},
        {'params': model.layer2.parameters(), 'lr': MIDDLE_LR},
        {'params': model.layer3.parameters(), 'lr': MIDDLE_LR},
        {'params': model.layer4.parameters(), 'lr': LATE_LR},
        {'params': model.fc.parameters(), 'lr': HEAD_LR},
    ]
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    labels, probabilities = [], []
    loss_sum, count = 0.0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda'):
            logits = model(x).flatten()
            loss = criterion(logits, y)
        labels.append(y.int().cpu().numpy())
        probabilities.append(torch.sigmoid(logits.float()).cpu().numpy())
        loss_sum += loss.item() * len(y)
        count += len(y)
    return np.concatenate(labels), np.concatenate(probabilities), loss_sum / count

def choose_threshold(y, p):
    fpr, tpr, thresholds = roc_curve(y, p)
    eligible = np.flatnonzero((tpr >= TARGET_SENSITIVITY) & np.isfinite(thresholds))
    if not len(eligible):
        raise ValueError('No finite threshold achieves target validation sensitivity')
    best = min(eligible, key=lambda idx: (fpr[idx], -thresholds[idx]))
    return float(thresholds[best])

def operating_metrics(y, p, threshold):
    predictions = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    return {
        'sensitivity': float(tp / (tp + fn)),
        'specificity': float(tn / (tn + fp)),
        'precision': float(precision_score(y, predictions, zero_division=0)),
        'f1': float(f1_score(y, predictions, zero_division=0)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


## Train and select the best model using validation AUROC

In [6]:
scaler = torch.amp.GradScaler('cuda')
best_auc = -1.0
best_epoch = None
stale = 0
history = []
optimizer = None

for epoch in range(1, MAX_EPOCHS + 1):
    stage = 'head' if epoch == 1 else 'full'
    if epoch in (1, 2):
        optimizer = set_training_stage(model, stage)
        print(f'Epoch {epoch}: switched to {stage} fine-tuning', flush=True)

    model.train()
    if stage == 'head':
        for layer in model.modules():
            if isinstance(layer, nn.modules.batchnorm._BatchNorm):
                layer.eval()

    epoch_loss, seen = 0.0, 0
    started = time.time()
    for x, y in loaders['train']:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda'):
            logits = model(x).flatten()
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item() * len(y)
        seen += len(y)

    y_val, p_val, val_loss = evaluate(model, loaders['val'])
    val_auc = float(roc_auc_score(y_val, p_val))
    val_ap = float(average_precision_score(y_val, p_val))
    row = {
        'epoch': epoch, 'stage': stage,
        'train_loss': epoch_loss / seen,
        'val_loss': val_loss,
        'val_auroc': val_auc,
        'val_pr_auc': val_ap,
        'seconds': round(time.time() - started, 1),
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_ROOT / 'history.csv', index=False)
    print(row, flush=True)

    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch
        stale = 0
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'stage': stage,
            'val_auroc': val_auc,
            'settings': {
                'variant': VARIANT, 'image_size': IMAGE_SIZE,
                'batch_size': BATCH_SIZE, 'loss': 'plain_bce',
                'optimizer': 'AdamW', 'weight_decay': WEIGHT_DECAY,
                'head_lr': HEAD_LR, 'early_lr': EARLY_LR,
                'middle_lr': MIDDLE_LR, 'late_lr': LATE_LR,
                'pretrained': 'ImageNet1K_V2', 'seed': SEED,
                'fine_tuning': 'head_epoch_1_then_full_layerwise',
            },
        }, OUTPUT_ROOT / 'best_model.pt')
    elif epoch >= 2:
        stale += 1

    if epoch >= 2 and stale >= PATIENCE:
        print('Early stopping after epoch', epoch, flush=True)
        break

print('Best validation epoch:', best_epoch, '| AUROC:', best_auc)


Epoch 1: switched to head fine-tuning
{'epoch': 1, 'stage': 'head', 'train_loss': 0.6731887734939593, 'val_loss': 0.6372353880011555, 'val_auroc': 0.7173805230052734, 'val_pr_auc': 0.4948956659383998, 'seconds': 236.3}
Epoch 2: switched to full fine-tuning
{'epoch': 2, 'stage': 'full', 'train_loss': 0.45431826909881695, 'val_loss': 0.34172869326017274, 'val_auroc': 0.8954501455190764, 'val_pr_auc': 0.8332621628003736, 'seconds': 670.4}
{'epoch': 3, 'stage': 'full', 'train_loss': 0.3120498973758773, 'val_loss': 0.356971061884318, 'val_auroc': 0.8974811940427372, 'val_pr_auc': 0.8368614999263368, 'seconds': 669.1}
{'epoch': 4, 'stage': 'full', 'train_loss': 0.17377172401779753, 'val_loss': 0.7610137500202152, 'val_auroc': 0.8918564950125734, 'val_pr_auc': 0.822115313271523, 'seconds': 670.0}
{'epoch': 5, 'stage': 'full', 'train_loss': 0.07570363914608015, 'val_loss': 0.4691114702019062, 'val_auroc': 0.8861949857975643, 'val_pr_auc': 0.8219456917583114, 'seconds': 668.4}
{'epoch': 6, 'sta

## Final validation and held-out test

In [7]:
checkpoint = torch.load(OUTPUT_ROOT / 'best_model.pt', map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model'])
model.eval()

val_y, val_p, val_loss = evaluate(model, loaders['val'])
threshold = choose_threshold(val_y, val_p)
val_results = {
    'model': 'ResNet-50',
    'variant': VARIANT,
    'loss': 'plain_bce',
    'balanced': True,
    'fine_tuning': 'head_epoch_1_then_full_layerwise',
    'optimizer': 'AdamW',
    'best_epoch': int(checkpoint['epoch']),
    'val_loss': float(val_loss),
    'val_auroc': float(roc_auc_score(val_y, val_p)),
    'val_pr_auc': float(average_precision_score(val_y, val_p)),
    'val_threshold_for_90pct_sensitivity': threshold,
    **{'val_' + key: value for key, value in operating_metrics(val_y, val_p, threshold).items()},
}
(OUTPUT_ROOT / 'validation_results.json').write_text(json.dumps(val_results, indent=2))
pd.DataFrame([val_results]).to_csv(OUTPUT_ROOT / 'validation_summary.csv', index=False)
pd.DataFrame({'label': val_y, 'probability': val_p}).to_csv(
    OUTPUT_ROOT / 'validation_predictions.csv', index=False
)
print('VALIDATION RESULTS')
print(json.dumps(val_results, indent=2), flush=True)

# The threshold is fixed from validation before evaluating the held-out test split.
test_y, test_p, test_loss = evaluate(model, loaders['test'])
test_results = {
    'model': 'ResNet-50',
    'variant': VARIANT,
    'loss': 'plain_bce',
    'balanced': True,
    'fine_tuning': 'head_epoch_1_then_full_layerwise',
    'optimizer': 'AdamW',
    'best_epoch': int(checkpoint['epoch']),
    'threshold_from_validation': threshold,
    'test_loss': float(test_loss),
    'test_auroc': float(roc_auc_score(test_y, test_p)),
    'test_pr_auc': float(average_precision_score(test_y, test_p)),
    **{'test_' + key: value for key, value in operating_metrics(test_y, test_p, threshold).items()},
}
(OUTPUT_ROOT / 'test_results.json').write_text(json.dumps(test_results, indent=2))
pd.DataFrame([test_results]).to_csv(OUTPUT_ROOT / 'test_summary.csv', index=False)
pd.DataFrame({'label': test_y, 'probability': test_p}).to_csv(
    OUTPUT_ROOT / 'test_predictions.csv', index=False
)
pd.DataFrame([{**val_results, **test_results}]).to_csv(
    OUTPUT_ROOT / 'combined_summary.csv', index=False
)
print('TEST RESULTS')
print(json.dumps(test_results, indent=2), flush=True)
print('All results saved to:', OUTPUT_ROOT)

del model, loaders, checkpoint
_ = gc.collect()
torch.cuda.empty_cache()


VALIDATION RESULTS
{
  "model": "ResNet-50",
  "variant": "nonbg",
  "loss": "plain_bce",
  "balanced": true,
  "fine_tuning": "head_epoch_1_then_full_layerwise",
  "optimizer": "AdamW",
  "best_epoch": 3,
  "val_loss": 0.356971061884318,
  "val_auroc": 0.8974811940427372,
  "val_pr_auc": 0.8368614999263368,
  "val_threshold_for_90pct_sensitivity": 0.22017472982406616,
  "val_sensitivity": 0.9000628535512256,
  "val_specificity": 0.616516762060507,
  "val_precision": 0.4328899637243047,
  "val_f1": 0.5846091038987549,
  "val_tn": 3016,
  "val_fp": 1876,
  "val_fn": 159,
  "val_tp": 1432
}
TEST RESULTS
{
  "model": "ResNet-50",
  "variant": "nonbg",
  "loss": "plain_bce",
  "balanced": true,
  "fine_tuning": "head_epoch_1_then_full_layerwise",
  "optimizer": "AdamW",
  "best_epoch": 3,
  "threshold_from_validation": 0.22017472982406616,
  "test_loss": 0.38378968360133503,
  "test_auroc": 0.8730912652096675,
  "test_pr_auc": 0.8065554113821856,
  "test_sensitivity": 0.865530303030303,
  

## Output files

`best_model.pt` is the validation-selected checkpoint. `history.csv` records training progress. `validation_results.json`, `validation_summary.csv`, and `validation_predictions.csv` contain validation outputs. `test_results.json`, `test_summary.csv`, and `test_predictions.csv` contain test outputs. `combined_summary.csv` contains both sets of metrics in one row.

**Kaggle:** Use **Save Version → Save & Run All**. The notebook trains from scratch and tests in the **same run**. Attach only the `02-preprocessing` output; no earlier BG/non-BG checkpoints are needed. The test is evaluated once after validation selection.